In [18]:
# Cell 1: Import Libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [19]:
# Cell 2: Load Dataset
df = pd.read_csv('../data/Language_translation_models_dataset.csv')
print(f"📊 Loaded {len(df)} records")
print(f"📋 Columns: {df.columns.tolist()}")

# Display first few rows
df.head()

📊 Loaded 2001 records
📋 Columns: ['Sentence_ID', 'Source_Language', 'Target_Language', 'Source_Text', 'Target_Text', 'Domain', 'Sentence_Length', 'Translation_Quality', 'Speaker_Region', 'Formality']


,Sentence_ID,Source_Language,Target_Language,Source_Text,Target_Text,Domain,Sentence_Length,Translation_Quality,Speaker_Region,Formality
0,S0001,English,Luganda,Thank you very much.,[Luganda translation] Webale nnyo,Daily Conversation,4,High,Eastern,Formal
1,S0002,English,Lusoga,The patient needs medicine,[Lusoga translation] The patient needs medicine,Health,4,High,Central,Informal
2,S0003,English,Luganda,Welcome,[Luganda translation] Welcome,Tourism,1,High,Western,Informal
3,S0004,English,Runyankole,School begins tomorrow,[Runyankole translation] School begins tomorrow,Education,3,High,West Nile,Formal
4,S0005,English,Luganda,Good morning,[Luganda translation] Wasuze otya,Business,2,High,Eastern,Formal


In [20]:
# Cell 3: Data Overview
print("📊 Dataset Overview:")
print("="*50)
print(f"   Total records: {len(df)}")
print(f"   Languages: {df['Target_Language'].nunique()}")
print(f"   Domains: {df['Domain'].nunique()}")
print(f"   Formality levels: {df['Formality'].nunique()}")

print("\n📊 Language Distribution:")
print(df['Target_Language'].value_counts())

print("\n📊 Domain Distribution:")
print(df['Domain'].value_counts())

📊 Dataset Overview:
   Total records: 2001
   Languages: 7
   Domains: 7
   Formality levels: 2

📊 Language Distribution:


Target_Language
Acholi        309
Runyankole    297
Luganda       293
Ateso         288
Lusoga        282
Lugbara       277
Rukiga        255
Name: count, dtype: int64

📊 Domain Distribution:
Domain
Health                505
Tourism               310
Agriculture           307
Education             295
Business              276
Daily Conversation    158
Government            150
Name: count, dtype: int64


In [21]:
# Cell 4: Clean Text Data (FIXED)
print("🧹 Cleaning Text Data...")
print("="*50)

def clean_text(text):
    """Clean text by removing extra spaces and special characters"""
    if isinstance(text, str):
        text = ' '.join(text.split())
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
        return text
    return text

def extract_translation(text):
    """
    Extract actual translation from dataset format.
    Handles: "[Rukiga translation] Good morning" → "Good morning"
    """
    if isinstance(text, str):
        # Check if it has the placeholder format
        if '[' in text and ']' in text:
            # Extract everything after the closing bracket
            parts = text.split(']')
            if len(parts) > 1:
                translation = parts[1].strip()
                if translation:
                    return translation
        
        # Also handle cases like "[Luganda translation] Webale nnyo"
        match = re.search(r'\]\s*(.+)$', text)
        if match:
            return match.group(1).strip()
        
        return text
    return text

# Apply cleaning
df['Source_Text_Cleaned'] = df['Source_Text'].apply(clean_text)
df['Target_Text_Cleaned'] = df['Target_Text'].apply(extract_translation)

# Show sample of before/after
print("📊 Sample translations BEFORE and AFTER:")
print("="*60)
for i in range(min(10, len(df))):
    print(f"\n{i+1}. BEFORE: {df['Target_Text'].iloc[i]}")
    print(f"   AFTER:  {df['Target_Text_Cleaned'].iloc[i]}")

# Remove rows where translation is still placeholder
df = df[~df['Target_Text_Cleaned'].str.contains(r'\[.*translation\]', na=False, case=False)]

# Remove duplicates
df = df.drop_duplicates(subset=['Source_Text', 'Target_Language'])

# Remove empty rows
df = df[df['Target_Text_Cleaned'].notna()]
df = df[df['Target_Text_Cleaned'].str.strip() != '']
df = df[df['Source_Text_Cleaned'].notna()]
df = df[df['Source_Text_Cleaned'].str.strip() != '']

print(f"\n✅ After cleaning: {len(df)} records remaining")
print(f"   Languages: {df['Target_Language'].nunique()}")
print(f"   Domains: {df['Domain'].nunique()}")

🧹 Cleaning Text Data...
📊 Sample translations BEFORE and AFTER:

1. BEFORE: [Luganda translation] Webale nnyo
   AFTER:  Webale nnyo

2. BEFORE: [Lusoga translation] The patient needs medicine
   AFTER:  The patient needs medicine

3. BEFORE: [Luganda translation] Welcome
   AFTER:  Welcome

4. BEFORE: [Runyankole translation] School begins tomorrow
   AFTER:  School begins tomorrow

5. BEFORE: [Luganda translation] Wasuze otya
   AFTER:  Wasuze otya

6. BEFORE: [Lusoga translation] Thank you
   AFTER:  Thank you

7. BEFORE: [Ateso translation] Open the door
   AFTER:  Open the door

8. BEFORE: [Runyankole translation] The road is bad
   AFTER:  The road is bad

9. BEFORE: [Lugbara translation] This park has many animals
   AFTER:  This park has many animals

10. BEFORE: [Acholi translation] This park has many animals
   AFTER:  This park has many animals

✅ After cleaning: 85 records remaining
   Languages: 7
   Domains: 7


In [22]:
# Cell 5: Encode Categorical Variables
le_language = LabelEncoder()
le_domain = LabelEncoder()
le_formality = LabelEncoder()

df['Language_encoded'] = le_language.fit_transform(df['Target_Language'])
df['Domain_encoded'] = le_domain.fit_transform(df['Domain'])
df['Formality_encoded'] = le_formality.fit_transform(df['Formality'])

print(f"✅ Encoded {len(le_language.classes_)} languages")
print(f"✅ Encoded {len(le_domain.classes_)} domains")
print(f"✅ Encoded {len(le_formality.classes_)} formality levels")

print("\n📊 Language Mapping:")
for i, lang in enumerate(le_language.classes_):
    print(f"   {lang}: {i}")

print("\n📊 Domain Mapping:")
for i, domain in enumerate(le_domain.classes_):
    print(f"   {domain}: {i}")

✅ Encoded 7 languages
✅ Encoded 7 domains
✅ Encoded 2 formality levels

📊 Language Mapping:
   Acholi: 0
   Ateso: 1
   Luganda: 2
   Lugbara: 3
   Lusoga: 4
   Rukiga: 5
   Runyankole: 6

📊 Domain Mapping:
   Agriculture: 0
   Business: 1
   Daily Conversation: 2
   Education: 3
   Government: 4
   Health: 5
   Tourism: 6


In [23]:
# Cell 6: TF-IDF Vectorization
vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2
)

source_vectors = vectorizer.fit_transform(df['Source_Text_Cleaned'])

print(f"✅ Vectorized {source_vectors.shape[0]} samples")
print(f"✅ Created {source_vectors.shape[1]} features")

✅ Vectorized 85 samples
✅ Created 38 features


In [24]:
# Cell 7: Build Language Models (FIXED)
print("🤖 Building Language Models...")
print("="*50)

language_models = {}

for lang in df['Target_Language'].unique():
    lang_df = df[df['Target_Language'] == lang]
    lang_vectors = vectorizer.transform(lang_df['Source_Text_Cleaned'])
    
    # Use Nearest Neighbors with more neighbors for better matching
    model = NearestNeighbors(n_neighbors=5, metric='cosine')
    model.fit(lang_vectors)
    
    language_models[lang] = {
        'model': model,
        'texts': lang_df['Target_Text_Cleaned'].values,  # ← USING CLEANED TRANSLATIONS
        'domains': lang_df['Domain'].values,
        'formality': lang_df['Formality'].values,
        'sources': lang_df['Source_Text'].values,
        'vectors': lang_vectors
    }
    
    print(f"   ✅ {lang}: {len(lang_df)} translations")

print(f"\n✅ Built models for {len(language_models)} languages")

🤖 Building Language Models...
   ✅ Luganda: 13 translations
   ✅ Lusoga: 12 translations
   ✅ Runyankole: 12 translations
   ✅ Ateso: 12 translations
   ✅ Lugbara: 12 translations
   ✅ Acholi: 12 translations
   ✅ Rukiga: 12 translations

✅ Built models for 7 languages


In [25]:
# Cell 8: Test Translation (FIXED)
print("🧪 Testing Translations...")
print("="*50)

def translate_text(text, target_language, domain=None, formality=None):
    """Test translation function"""
    if target_language not in language_models:
        return {'success': False, 'error': f'Language "{target_language}" not supported'}
    
    # Clean text
    text = clean_text(text)
    
    if not text:
        return {'success': False, 'error': 'No text to translate'}
    
    # Vectorize
    text_vector = vectorizer.transform([text])
    
    # Find nearest translations
    lang_model = language_models[target_language]
    distances, indices = lang_model['model'].kneighbors(text_vector)
    
    # Get matches - WITH CLEANED TRANSLATIONS
    matches = []
    for idx in indices[0]:
        if idx < len(lang_model['texts']):
            matches.append({
                'translation': lang_model['texts'][idx],
                'domain': lang_model['domains'][idx],
                'formality': lang_model['formality'][idx],
                'distance': float(distances[0][list(indices[0]).index(idx)])
            })
    
    if not matches:
        return {'success': False, 'error': 'No translation found'}
    
    # Filter by domain
    if domain and domain != 'All':
        matches = [m for m in matches if m['domain'] == domain]
        if not matches:
            return {'success': False, 'error': f'No translations in domain: {domain}'}
    
    # Filter by formality
    if formality and formality != 'All':
        matches = [m for m in matches if m['formality'] == formality]
        if not matches:
            return {'success': False, 'error': f'No translations with formality: {formality}'}
    
    return matches[0]

# Test translations
test_cases = [
    ("Thank you", "Luganda"),
    ("Good morning", "Rukiga"),
    ("Good morning", "Acholi"),
    ("Good morning", "Luganda"),
    ("Welcome", "Luganda"),
    ("Hello", "Lusoga"),
    ("Goodbye", "Runyankole"),
    ("Open the door", "Ateso"),
    ("The patient needs medicine", "Lusoga"),
    ("The crops are ready", "Lugbara"),
]

print("📊 Test Results:")
print("="*60)

for text, lang in test_cases:
    print(f"\n🔍 Input: '{text}' → {lang}")
    result = translate_text(text, lang)
    
    if isinstance(result, dict):
        print(f"   ✅ Translation: {result['translation']}")
        print(f"   📋 Domain: {result['domain']}")
        print(f"   📊 Confidence: {(1 - result['distance'])*100:.1f}%")
    else:
        print(f"   ❌ {result}")

🧪 Testing Translations...
📊 Test Results:

🔍 Input: 'Thank you' → Luganda


   ✅ Translation: Thank you
   📋 Domain: Health
   📊 Confidence: 100.0%

🔍 Input: 'Good morning' → Rukiga
   ✅ Translation: Good morning
   📋 Domain: Tourism
   📊 Confidence: 100.0%

🔍 Input: 'Good morning' → Acholi
   ✅ Translation: Good morning
   📋 Domain: Daily Conversation
   📊 Confidence: 100.0%

🔍 Input: 'Good morning' → Luganda
   ✅ Translation: Wasuze otya
   📋 Domain: Business
   📊 Confidence: 100.0%

🔍 Input: 'Welcome' → Luganda
   ✅ Translation: Welcome
   📋 Domain: Tourism
   📊 Confidence: 100.0%

🔍 Input: 'Hello' → Lusoga
   ✅ Translation: Open the door
   📋 Domain: Tourism
   📊 Confidence: 0.0%

🔍 Input: 'Goodbye' → Runyankole
   ✅ Translation: This park has many animals
   📋 Domain: Tourism
   📊 Confidence: 0.0%

🔍 Input: 'Open the door' → Ateso
   ✅ Translation: Open the door
   📋 Domain: Business
   📊 Confidence: 100.0%

🔍 Input: 'The patient needs medicine' → Lusoga
   ✅ Translation: The patient needs medicine
   📋 Domain: Health
   📊 Confidence: 100.0%

🔍 Input: 'Th

In [26]:
# Cell 9: Model Evaluation
print("📊 Evaluating Model...")
print("="*50)

sample_df = df.sample(min(500, len(df)), random_state=42)

correct = 0
total = 0
results = []

print(f"🔍 Testing on {len(sample_df)} samples...")

for idx, row in sample_df.iterrows():
    text = row['Source_Text_Cleaned']
    target = row['Target_Language']
    actual = row['Target_Text_Cleaned']
    
    result = translate_text(text, target)
    
    if isinstance(result, dict):
        predicted = result['translation']
        if predicted == actual:
            correct += 1
        total += 1
        results.append({
            'source': text,
            'actual': actual,
            'predicted': predicted,
            'correct': predicted == actual
        })

accuracy = correct / total if total > 0 else 0

print(f"\n📊 Results:")
print(f"   Samples evaluated: {total}")
print(f"   Correct translations: {correct}")
print(f"   Accuracy: {accuracy*100:.2f}%")

📊 Evaluating Model...
🔍 Testing on 85 samples...

📊 Results:
   Samples evaluated: 85
   Correct translations: 84
   Accuracy: 98.82%


In [30]:
# Cell 10: Save Model
print("💾 Saving Model...")
print("="*50)

os.makedirs('../models/saved_models', exist_ok=True)

joblib.dump(vectorizer, '../models/saved_models/vectorizer.pkl')
joblib.dump(language_models, '../models/saved_models/language_models.pkl')
joblib.dump(le_language, '../models/saved_models/le_language.pkl')
joblib.dump(le_domain, '../models/saved_models/le_domain.pkl')
joblib.dump(le_formality, '../models/saved_models/le_formality.pkl')
joblib.dump(df, '../models/saved_models/training_data.pkl')

print("✅ Model saved to models/saved_models/")

# List saved files
print("\n📁 Saved Files:")
for file in os.listdir('../models/saved_models/'):
    size = os.path.getsize(f'../models/saved_models/{file}')
    print(f"   ✅ {file} ({size:,} bytes)")

💾 Saving Model...
✅ Model saved to models/saved_models/

📁 Saved Files:
   ✅ fix_translations.py (1,343 bytes)
   ✅ language_models.pkl (21,614 bytes)
   ✅ le_domain.pkl (563 bytes)
   ✅ le_formality.pkl (493 bytes)
   ✅ le_language.pkl (541 bytes)
   ✅ model_report.json (1,027 bytes)
   ✅ training_data.pkl (16,499 bytes)
   ✅ vectorizer.pkl (2,317 bytes)


In [31]:
# Cell 11: Generate Model Report
print("📝 Generating Model Report...")
print("="*50)

report = {
    'model_name': 'TF-IDF + Nearest Neighbors',
    'total_samples': len(df),
    'languages': df['Target_Language'].unique().tolist(),
    'domains': df['Domain'].unique().tolist(),
    'formality_levels': df['Formality'].unique().tolist(),
    'language_distribution': df['Target_Language'].value_counts().to_dict(),
    'domain_distribution': df['Domain'].value_counts().to_dict(),
    'accuracy': accuracy,
    'samples_evaluated': len(results),
    'correct_predictions': sum(1 for r in results if r['correct']),
    'incorrect_predictions': sum(1 for r in results if not r['correct']),
    'features': source_vectors.shape[1],
    'training_date': datetime.now().isoformat(),
    'model_type': 'TF-IDF + Nearest Neighbors'
}

with open('../models/saved_models/model_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("✅ Model report saved!")

📝 Generating Model Report...
✅ Model report saved!


In [29]:
# Cell 12: Training Summary
print("="*70)
print("🎉 MODEL TRAINING COMPLETE!")
print("="*70)

print(f"\n📊 Summary:")
print(f"   Total samples: {len(df):,}")
print(f"   Languages: {len(language_models)}")
print(f"   Domains: {len(le_domain.classes_)}")
print(f"   Features: {source_vectors.shape[1]:,}")
print(f"   Accuracy: {accuracy*100:.2f}%")

print(f"\n📁 Files saved:")
print(f"   ✅ models/saved_models/vectorizer.pkl")
print(f"   ✅ models/saved_models/language_models.pkl")
print(f"   ✅ models/saved_models/le_language.pkl")
print(f"   ✅ models/saved_models/le_domain.pkl")
print(f"   ✅ models/saved_models/le_formality.pkl")
print(f"   ✅ models/saved_models/training_data.pkl")
print(f"   ✅ models/saved_models/model_report.json")

🎉 MODEL TRAINING COMPLETE!

📊 Summary:
   Total samples: 85
   Languages: 7
   Domains: 7
   Features: 38
   Accuracy: 98.82%

📁 Files saved:
   ✅ models/saved_models/vectorizer.pkl
   ✅ models/saved_models/language_models.pkl
   ✅ models/saved_models/le_language.pkl
   ✅ models/saved_models/le_domain.pkl
   ✅ models/saved_models/le_formality.pkl
   ✅ models/saved_models/training_data.pkl
   ✅ models/saved_models/model_report.json


In [15]:
# Cell 14: Model Summary
print("="*70)
print("🎉 MODEL TRAINING COMPLETE!")
print("="*70)

print(f"\n📊 Summary:")
print(f"   Total samples: {len(df):,}")
print(f"   Languages: {len(language_models)}")
print(f"   Domains: {len(le_domain.classes_)}")
print(f"   Features: {source_vectors.shape[1]:,}")
print(f"   Accuracy: {accuracy*100:.2f}%")
print(f"   Evaluated: {len(results)} samples")
print(f"   Correct: {correct_count}")
print(f"   Incorrect: {incorrect_count}")

print(f"\n📁 Files saved:")
print(f"   ✅ models/saved_models/vectorizer.pkl")
print(f"   ✅ models/saved_models/language_models.pkl")
print(f"   ✅ models/saved_models/le_language.pkl")
print(f"   ✅ models/saved_models/le_domain.pkl")
print(f"   ✅ models/saved_models/le_formality.pkl")
print(f"   ✅ models/saved_models/training_data.pkl")
print(f"   ✅ models/saved_models/model_report.json")

🎉 MODEL TRAINING COMPLETE!

📊 Summary:
   Total samples: 1,992
   Languages: 7
   Domains: 7
   Features: 38
   Accuracy: 0.00%
   Evaluated: 500 samples
   Correct: 0
   Incorrect: 500

📁 Files saved:
   ✅ models/saved_models/vectorizer.pkl
   ✅ models/saved_models/language_models.pkl
   ✅ models/saved_models/le_language.pkl
   ✅ models/saved_models/le_domain.pkl
   ✅ models/saved_models/le_formality.pkl
   ✅ models/saved_models/training_data.pkl
   ✅ models/saved_models/model_report.json


In [ ]:
# Check language distribution
print("📊 Language Distribution:")
print("="*50)
lang_counts = df['Target_Language'].value_counts()
print(lang_counts)

# Show samples per language
print("\n📋 Sample translations per language:")
for lang in df['Target_Language'].unique():
    lang_df = df[df['Target_Language'] == lang]
    print(f"\n{lang}: {len(lang_df)} translations")
    print(f"   Sample: {lang_df['Source_Text'].iloc[0]} → {lang_df['Target_Text'].iloc[0]}")